# Day 27 — Practice SOLUTIONS
### The Central Limit Theorem · Module 2 · Topic 2.7

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it tests | Count |
|---|---|---|
| **A** | Predict the output | 8 |
| **B** | Spot & fix the bug | 4 |
| **C** | Write the code | 5 |
| **D** | Challenge | 2 |

**How to work:** for Part A, write your answer down *before* running anything.

Three questions to keep asking yourself:

- Am I talking about the **data**, or about the **mean of the data**?
- Did I divide by **√n**, or by n?
- Does the theorem actually **apply** here — finite σ, independent observations, enough n?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

rng = np.random.default_rng(27)
print("ready")

---
# Part A — Predict the output

Write your answer first. Then run the cell.

### A1

```python
pop = rng.exponential(5, 500_000)
print(round(stats.skew(pop), 2))

means = rng.exponential(5, (100_000, 100)).mean(axis=1)
print(round(stats.skew(means), 2))
```

Two numbers. What are they, and what is the relationship between them?

In [ ]:
pop = rng.exponential(5, 500_000)
print(round(stats.skew(pop), 2))

means = rng.exponential(5, (100_000, 100)).mean(axis=1)
print(round(stats.skew(means), 2))

**About `2.0` and about `0.2`.**

An exponential always has a skew of exactly 2, whatever its mean — note that changing the mean
from 10 to 5 did not change the skew at all.

The second is the first divided by √100 = 10. **The skew of the sample mean is the population
skew over √n**, exactly.

In [ ]:
print(f"population skew          {stats.skew(pop):.3f}   (exactly 2 for any exponential)")
print(f"skew of the mean of 100  {stats.skew(means):.3f}")
print(f"2 / sqrt(100)            {2/np.sqrt(100):.3f}")
print()
for n in [4, 25, 100, 400]:
    m = rng.exponential(5, (60_000, n)).mean(axis=1)
    print(f"  n={n:>4}: observed {stats.skew(m):.3f}   predicted {2/np.sqrt(n):.3f}")

### A2

```python
data = rng.normal(100, 15, 400)
print(round(data.std(ddof=1), 2))
print(round(data.std(ddof=1) / np.sqrt(400), 2))
```

Which of these two is the standard error, and which would you quote as "the spread of exam
marks"?

In [ ]:
data = rng.normal(100, 15, 400)
print(round(data.std(ddof=1), 2))
print(round(data.std(ddof=1) / np.sqrt(400), 2))

**About `15` and about `0.75`.**

- **15** is σ — the spread of the **marks**. That is what you quote when describing students.
- **0.75** is the standard error — how far the **class average** is likely to land from the
  true average. That is what you quote when describing how well you measured.

Collect 10,000 students and the first number stays at 15. The second drops to 0.15.

Reporting the standard error when someone asked about the spread of marks makes your data look
twenty times tighter than it is. This is the second most common error in the topic.

In [ ]:
print(f"{'n':>8} {'sd of marks':>14} {'SE of the mean':>17}")
for n in [100, 400, 2500, 10_000]:
    d = rng.normal(100, 15, n)
    print(f"{n:>8} {d.std(ddof=1):>14.2f} {d.std(ddof=1)/np.sqrt(n):>17.3f}")
print()
print("sigma describes PEOPLE and does not move.")
print("SE describes the AVERAGE and shrinks with sqrt(n).")

### A3

A population has σ = 20. You want a standard error of 1.

```python
print((20 / 1) ** 2)
print((20 / 0.5) ** 2)
```

What do these two numbers mean, and what does the jump between them tell you?

In [ ]:
print((20 / 1) ** 2)
print((20 / 0.5) ** 2)

**`400` and `1600`.**

To get a standard error of 1 you need n = 400. To halve it to 0.5 you need n = 1600 — **four
times the data for twice the precision**.

Rearranging SE = σ/√n gives **n = (σ/SE)²**, which is the formula behind every "how many users
do we need?" conversation. The square is what makes precision expensive.

In [ ]:
print(f"{'target SE':>11} {'n needed':>10}")
for se in [4, 2, 1, 0.5, 0.25]:
    print(f"{se:>11} {(20/se)**2:>10.0f}")
print()
print("Each halving of the error costs 4x the data.")
print("Going from SE=4 to SE=0.25 is 256x the sample size.")

### A4

```python
d = stats.cauchy.rvs(size=100_000, random_state=1)
m = stats.cauchy.rvs(size=(100_000, 1000), random_state=1).mean(axis=1)
print(round(np.percentile(d, 75) - np.percentile(d, 25), 2))
print(round(np.percentile(m, 75) - np.percentile(m, 25), 2))
```

The second line averages a **thousand** observations. How much narrower is it?

In [ ]:
d = stats.cauchy.rvs(size=100_000, random_state=1)
m = stats.cauchy.rvs(size=(100_000, 1000), random_state=1).mean(axis=1)
print(round(np.percentile(d, 75) - np.percentile(d, 25), 2))
print(round(np.percentile(m, 75) - np.percentile(m, 25), 2))

**Both about `2.0`. It is not narrower at all.**

A Cauchy has infinite variance, so the theorem's one real condition fails. The mean of n Cauchy
samples is *itself Cauchy, with exactly the same spread* — averaging a thousand of them buys you
literally nothing.

Compare a normal, where 1,000 observations would have narrowed the spread by a factor of √1000
≈ 32.

In [ ]:
print(f"{'n':>7} {'Cauchy IQR':>13} {'Normal IQR':>13}")
for n in [1, 10, 100, 1000]:
    c = stats.cauchy.rvs(size=(40_000, n), random_state=n).mean(axis=1)
    g = np.random.default_rng(n).normal(0, 1, (40_000, n)).mean(axis=1)
    print(f"{n:>7} {np.percentile(c,75)-np.percentile(c,25):>13.3f} "
          f"{np.percentile(g,75)-np.percentile(g,25):>13.3f}")
print()
print("The normal column shrinks as 1/sqrt(n). The Cauchy column does not shrink at all.")

### A5

```python
pop = rng.lognormal(0, 1, 500_000)
print(round(stats.shapiro(rng.choice(pop, 500)).pvalue, 6))
means = np.array([rng.choice(pop, 500).mean() for _ in range(500)])
print(round(stats.shapiro(means).pvalue, 4))
```

One of these tests passes and one fails. Which, and does the failure matter?

In [ ]:
pop = rng.lognormal(0, 1, 500_000)
print(round(stats.shapiro(rng.choice(pop, 500)).pvalue, 6))
means = np.array([rng.choice(pop, 500).mean() for _ in range(500)])
print(round(stats.shapiro(means).pvalue, 4))

**The first fails (p ≈ 0), the second passes.**

The first tested the **raw data** for normality. Lognormal data is not normal and never will
be, so that test fails and will keep failing however much data you collect.

The second tested the **sample means**, which is what the CLT is about — and they pass
comfortably.

**Does the failure matter? No.** It was never relevant. If you find yourself running a
normality test on raw data before building a confidence interval for its mean, you are testing
the wrong thing.

In [ ]:
raw   = rng.choice(pop, 500)
means = np.array([rng.choice(pop, 500).mean() for _ in range(500)])

print(f"raw data   : skew {stats.skew(raw):6.2f}   Shapiro p = {stats.shapiro(raw).pvalue:.2e}")
print(f"the means  : skew {stats.skew(means):6.2f}   Shapiro p = {stats.shapiro(means).pvalue:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 2.8))
axes[0].hist(raw, bins=60, color="#E8674C"); axes[0].set_title("the DATA - fails, correctly")
axes[1].hist(means, bins=40, color="#3E7CB1"); axes[1].set_title("the MEANS - passes")
for ax in axes: ax.set_yticks([])
plt.tight_layout(); plt.show()

### A6

```python
pop = rng.uniform(0, 1, 500_000)
m = rng.uniform(0, 1, (200_000, 12)).sum(axis=1)
print(round(m.mean(), 2), round(m.std(), 3))
```

You met this exact trick on Day 26. What comes out, and why those values?

In [ ]:
pop = rng.uniform(0, 1, 500_000)
m = rng.uniform(0, 1, (200_000, 12)).sum(axis=1)
print(round(m.mean(), 2), round(m.std(), 3))

**About `6.0` and about `1.0`.**

A U(0,1) has mean 1/2 and variance 1/12. Sum twelve independent ones and the means add
(12 × 1/2 = 6) and the **variances** add (12 × 1/12 = 1), so the sd is exactly 1.

So `sum(12 uniforms) − 6` is an almost perfect standard normal — which is what Day 26's Part D
discovered, and the CLT is why it works.

Note the general rule hiding here: **variances add, standard deviations do not.** That is where
the √n in σ/√n ultimately comes from.

In [ ]:
print(f"mean {m.mean():.4f}  (theory 12 x 0.5 = 6)")
print(f"var  {m.var():.4f}  (theory 12 x 1/12 = 1)")
print(f"sd   {m.std():.4f}  (theory sqrt(1) = 1)")
print(f"skew {stats.skew(m):+.4f}   kurtosis {stats.kurtosis(m):+.4f}   (normal: 0, 0)")
print()
print("Variances add. Standard deviations do not - that is where sqrt(n) comes from.")

### A7

```python
pop = rng.exponential(10, 200_000)
a = pop.std() / np.sqrt(100)
b = pop.std() / 100
print(round(a, 3), round(b, 3))
```

One of these is the standard error. Which, and how wrong is the other?

In [ ]:
pop = rng.exponential(10, 200_000)
a = pop.std() / np.sqrt(100)
b = pop.std() / 100
print(round(a, 3), round(b, 3))

**`a` ≈ 1.0 is the standard error. `b` ≈ 0.1 is wrong by a factor of 10.**

Dividing by n instead of √n is the single most common slip in this topic, and it always errs in
the dangerous direction — it makes your confidence intervals **too narrow** and your p-values
**too small**.

At n = 100 it is off by 10×. At n = 10,000 it is off by 100×. The bigger your study, the worse
the mistake gets.

In [ ]:
print(f"{'n':>8} {'sd/sqrt(n)':>13} {'sd/n':>10} {'factor':>9}")
for n in [25, 100, 1000, 10_000]:
    print(f"{n:>8} {10/np.sqrt(n):>13.4f} {10/n:>10.4f} {np.sqrt(n):>8.0f}x")
print()
print("The error grows with sqrt(n) - it gets worse the more data you collect.")

### A8

A survey of 1,000 people reports "45% support, margin of error ±3%".

```python
p = 0.45
print(round(1.96 * np.sqrt(p * (1 - p) / 1000) * 100, 1))
print(round((1.96 * np.sqrt(0.25) / 0.015) ** 2))
```

What are these two numbers?

In [ ]:
p = 0.45
print(round(1.96 * np.sqrt(p * (1 - p) / 1000) * 100, 1))
print(round((1.96 * np.sqrt(0.25) / 0.015) ** 2))

**About `3.1` and about `4269`.**

The first *is* the ±3% — it is just σ/√n with σ = √(p(1−p)), dressed up. Every "margin of
error" you have ever read is this calculation, and n ≈ 1,000 is why the answer is almost always
about ±3%.

The second is how many people you would need to halve it to ±1.5%: roughly **4,270**. Four
times the sample for twice the precision, exactly as the √n law says.

That is why polls are almost always around 1,000 people. It is the point where the margin is
tolerable and the cost is not yet absurd.

In [ ]:
print(f"{'n':>8} {'margin of error':>18}")
for n in [250, 500, 1000, 2000, 4269, 10_000]:
    print(f"{n:>8} {1.96*np.sqrt(0.25/n)*100:>17.2f}%")
print()
print("p(1-p) peaks at p=0.5, so sqrt(0.25)=0.5 is the worst case -")
print("which is why pollsters quote it regardless of the actual result.")

---
# Part B — Spot & fix the bug

Each cell runs without error and gives a **wrong** answer. Find the bug, say what the wrong
number means, and write the fix.

### B1 — the classic

A team measures page load times for 400 sessions and reports a confidence interval.

In [ ]:
rng_b1 = np.random.default_rng(401)
load = rng_b1.exponential(2.0, 400)        # seconds

mean = load.mean()
se   = load.std(ddof=1) / 400
print(f"mean load {mean:.3f}s   95% CI {mean-1.96*se:.3f} to {mean+1.96*se:.3f}")

**Bug:** `se = sd / 400` divides by n instead of **√n**.

**The wrong number** is a standard error 20× too small, so the interval is 20× too narrow. It
comes out roughly ±0.01s when the truth is roughly ±0.2s — an interval so tight it would
declare almost any comparison significant.

**Fix:** `se = load.std(ddof=1) / np.sqrt(400)`.

A quick sanity check that would have caught it: the data has an sd of about 2 seconds. An
interval of ±0.01s on 400 noisy observations is not plausible — you cannot measure an average
to a hundredth of a second from data that varies by whole seconds.

In [ ]:
rng_b1 = np.random.default_rng(401)
load = rng_b1.exponential(2.0, 400)
sd = load.std(ddof=1)

wrong, right = sd/400, sd/np.sqrt(400)
print(f"sd of the data        {sd:.4f}")
print(f"wrong  sd/n           {wrong:.4f}   -> CI width {2*1.96*wrong:.4f}s")
print(f"right  sd/sqrt(n)     {right:.4f}   -> CI width {2*1.96*right:.4f}s")
print(f"off by a factor of    {np.sqrt(400):.0f}")
print()
#  check it against the truth by simulation
sim = rng_b1.exponential(2.0, (20_000, 400)).mean(axis=1)
print(f"the real sd of a mean of 400 is {sim.std():.4f}  <- matches sd/sqrt(n)")

### B2 — the wrong distribution

A clothing retailer reports the range customers' heights fall in.

In [ ]:
rng_b2 = np.random.default_rng(402)
heights = rng_b2.normal(170, 9, 500)       # cm

m  = heights.mean()
se = heights.std(ddof=1) / np.sqrt(500)
print(f"95% of customers are between {m-1.96*se:.1f} and {m+1.96*se:.1f} cm")

**Bug:** the sentence says "95% of **customers**", but the interval was built from the
**standard error** — which describes the *average*, not the people.

**The wrong number** claims 95% of customers are between about 169 and 171 cm. That is a 2 cm
range for an entire customer base, which would be absurd for anyone buying clothing sizes.

**Fix:** decide which question you are answering.

- *Where do 95% of **customers** fall?* → use σ: `m ± 1.96 * sd` → roughly 152 to 188 cm.
- *How well do we know the **average**?* → use SE: `m ± 1.96 * se` → roughly 169 to 171 cm.

Both are correct statements. They answer completely different questions, and only one of them
is useful for sizing a clothing range.

In [ ]:
rng_b2 = np.random.default_rng(402)
heights = rng_b2.normal(170, 9, 500)
m, sd = heights.mean(), heights.std(ddof=1)
se = sd/np.sqrt(500)

print(f"Where 95% of CUSTOMERS fall:  {m-1.96*sd:.1f} to {m+1.96*sd:.1f} cm   (uses sigma)")
print(f"How well we know the AVERAGE: {m-1.96*se:.1f} to {m+1.96*se:.1f} cm   (uses SE)")
print()
inside = ((heights > m-1.96*sd) & (heights < m+1.96*sd)).mean()
print(f"check: {inside:.1%} of the sample really does fall in the first range")
inside_se = ((heights > m-1.96*se) & (heights < m+1.96*se)).mean()
print(f"       only {inside_se:.1%} falls in the second - it was never meant to contain people")

### B3 — the hidden dependence

A factory takes a temperature reading every minute for 200 minutes and reports the average.

In [ ]:
rng_b3 = np.random.default_rng(403)
n, rho = 200, 0.9

#  temperature drifts: each minute is close to the last
e = rng_b3.normal(0, np.sqrt(1 - rho**2), n)
temp = np.zeros(n); temp[0] = rng_b3.normal()
for t in range(1, n):
    temp[t] = rho * temp[t-1] + e[t]
temp = 300 + 2 * temp                      # degrees

se = temp.std(ddof=1) / np.sqrt(n)
print(f"mean {temp.mean():.3f} C   95% CI +/- {1.96*se:.3f}")

**Bug:** σ/√n assumes the observations are **independent**. Consecutive temperature readings
one minute apart are not — each one largely repeats the last.

**The wrong number** is an interval far too narrow. The formula treats 200 correlated readings
as 200 independent ones; at ρ = 0.9 they are worth roughly **11** independent readings, so the
true standard error is about **4.2× larger**.

**Fix:** there is no fix that keeps the same data. Either

- compute an **effective sample size** and use σ/√n_eff, or
- sample further apart in time so the readings really are independent, or
- use a method built for correlated data (block bootstrap, or an explicit time-series model).

The important part is noticing. Nothing in the output warns you, and this is the most common
way a real analysis quietly produces over-confident intervals.

In [ ]:
rng_b3 = np.random.default_rng(403)
n, rho = 200, 0.9

def ar1(n, rho, rng):
    e = rng.normal(0, np.sqrt(1-rho**2), n)
    x = np.zeros(n); x[0] = rng.normal()
    for t in range(1, n):
        x[t] = rho*x[t-1] + e[t]
    return x

#  what the naive formula claims
temp = 300 + 2*ar1(n, rho, rng_b3)
naive = temp.std(ddof=1)/np.sqrt(n)

#  the exact answer for AR(1) - no simulation noise
def exact_sd_of_mean(n, rho):
    k = np.arange(1, n)
    return np.sqrt((1/n) * (1 + 2*np.sum((1 - k/n) * rho**k)))

#  2 is the series' sd here, so scale the unit-variance result
true_se = 2 * exact_sd_of_mean(n, rho)
print(f"naive SE  (sigma/sqrt(n))      {naive:.4f}")
print(f"true SE   (exact for AR(1))    {true_se:.4f}")
print(f"the naive interval is {true_se/naive:.1f}x too narrow")
print()
print(f"effective sample size: {1/exact_sd_of_mean(n, rho)**2:.0f}, not {n}")
print()
#  the effect depends entirely on how strong the correlation is
print(f"{'rho':>6} {'inflation':>11} {'n_eff out of 200':>18}")
for r in [0.0, 0.3, 0.6, 0.9, 0.95]:
    sd_ = exact_sd_of_mean(n, r)
    print(f"{r:>6} {sd_*np.sqrt(n):>10.2f}x {1/sd_**2:>18.0f}")

### B4 — "the CLT says it's fine"

An analyst reports the average value of insurance claims.

In [ ]:
rng_b4 = np.random.default_rng(404)
#  claim sizes: a Pareto tail so heavy that the variance is infinite
claims = (rng_b4.pareto(1.2, 2000) + 1) * 1000

m  = claims.mean()
se = claims.std(ddof=1) / np.sqrt(2000)
print(f"average claim  {m:,.0f}   95% CI {m-1.96*se:,.0f} to {m+1.96*se:,.0f}")
print("n = 2000, well over 30, so the CLT applies.")

**Bug:** "n = 2000, so the CLT applies" skips the theorem's one real condition — a **finite σ**.
A Pareto with α = 1.2 has a finite mean but an **infinite variance**, so σ/√n is meaningless
however large n is.

**The wrong number** is an interval that will not reproduce. Run the same analysis on next
year's claims and the "average" can move by far more than the interval allowed, because a
single large claim can shift the mean arbitrarily.

**Fix:** diagnose first.

- Plot the **running mean**. If it is still jumping after thousands of points, stop.
- Report the **median**, or a trimmed mean, and say why.
- If you must report a mean, get its uncertainty from a **bootstrap** rather than a formula —
  and expect it to be wide and asymmetric.

"Average claim size" is a well-known way to be confidently wrong in insurance, for exactly this
reason.

In [ ]:
rng_b4 = np.random.default_rng(404)
claims = (rng_b4.pareto(1.2, 2000) + 1) * 1000

#  1. does the mean settle?
run = np.cumsum(claims)/np.arange(1, len(claims)+1)
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(run, color="#E8674C", lw=1.4)
ax.set_xlabel("claims collected"); ax.set_ylabel("running mean")
ax.set_title("Still jumping after 2,000 claims - sigma is not finite")
plt.show()

tail = run[len(run)//2:]
iqr = np.percentile(claims,75)-np.percentile(claims,25)
print(f"drift over the last half: {(tail.max()-tail.min())/iqr:.2f} of an IQR  (healthy data: ~0.01)")
print()

#  2. how unstable is the 'average' across repeats?
reps = np.array([((rng_b4.pareto(1.2, 2000)+1)*1000).mean() for _ in range(2000)])
se_formula = claims.std(ddof=1)/np.sqrt(2000)
print(f"the formula's SE          {se_formula:>12,.0f}")
print(f"actual sd of the mean     {reps.std():>12,.0f}")
print(f"actual 95% range of means {np.percentile(reps,2.5):>12,.0f} to {np.percentile(reps,97.5):,.0f}")
print()
print(f"median claim (stable)     {np.median(claims):>12,.0f}")
med = np.array([np.median((rng_b4.pareto(1.2,2000)+1)*1000) for _ in range(2000)])
print(f"  its 95% range           {np.percentile(med,2.5):>12,.0f} to {np.percentile(med,97.5):,.0f}")
print()
print("The median is a well-behaved summary here. The mean is not.")

---
# Part C — Write the code

Write the code yourself. Then say the answer in plain English.

### C1 — see it for yourself

Pick any population you like that is **not** normal.

1. Draw 200,000 values and plot them. Report the mean, sd and skew.
2. For n = 2, 10, 50, draw 200,000 sample means each and plot them.
3. Check the sd of each against σ/√n and the skew against skew/√n.
4. At which n would you be comfortable using a normal approximation, and why?

In [ ]:
rng_c1 = np.random.default_rng(271)

#  a bimodal population - about as un-normal as it gets
def draw(k):
    return np.where(rng_c1.random(k) < 0.5,
                    rng_c1.normal(-3, 0.5, k), rng_c1.normal(3, 0.5, k))

pop = draw(200_000)
print(f"population: mean {pop.mean():.3f}  sd {pop.std():.3f}  skew {stats.skew(pop):+.3f}")
print()
print(f"{'n':>4} {'sd of means':>13} {'sigma/sqrt(n)':>15} {'skew':>9} {'kurtosis':>10}")

fig, axes = plt.subplots(1, 4, figsize=(13, 2.7))
axes[0].hist(pop, bins=80, color="#E8674C"); axes[0].set_title("population"); axes[0].set_yticks([])
for ax, n in zip(axes[1:], [2, 10, 50]):
    m = draw(200_000 * n).reshape(200_000, n).mean(axis=1)
    print(f"{n:>4} {m.std():>13.4f} {pop.std()/np.sqrt(n):>15.4f} "
          f"{stats.skew(m):>+9.3f} {stats.kurtosis(m):>+10.3f}")
    ax.hist(m, bins=70, color="#3E7CB1", density=True)
    g = np.linspace(m.min(), m.max(), 300)
    ax.plot(g, stats.norm.pdf(g, m.mean(), m.std()), color="#152238", lw=2)
    ax.set_title(f"mean of {n}"); ax.set_yticks([])
plt.tight_layout(); plt.show()

The bimodal population has **skew 0 already** — it is symmetric, just not remotely normal. So
the skew column tells you nothing here, and **kurtosis** is the number to watch instead.

At n = 2 the means are still visibly two-humped. By n = 10 the humps have merged. By n = 50 the
normal curve fits well and kurtosis is near 0.

**Which n?** Around 10–20 for this population. And notice that the answer came from looking at
the *shape*, not from a rule — the skew-based rule would have said n = 1, which is clearly
wrong here.

That is an honest limitation of the skew rule from the slides: it catches *skewed* populations,
not every kind of non-normality. For a symmetric but strange population, excess kurtosis is the
better guide.

### C2 — how big a sample do I actually need?

Write a function `needed_n(population_sample, target_skew=0.3)` that estimates the sample size
required for the CLT to work well on a given population.

1. Use the rule n ≈ (skew / target)².
2. Test it on an exponential (skew 2) and a lognormal (skew ≈ 4.7).
3. **Then check your own answer by simulation** — measure the real coverage at the n your
   function recommends.
4. Does the rule deliver what it promises?

In [ ]:
rng_c2 = np.random.default_rng(272)

def needed_n(sample, target_skew=0.3):
    sk = stats.skew(np.asarray(sample, dtype=float))
    return max(2, int(np.ceil((abs(sk) / target_skew) ** 2)))

def coverage(sampler, n, mu, trials=4000):
    hits = 0
    for _ in range(trials):
        s = sampler(n)
        se = s.std(ddof=1) / np.sqrt(n)
        hits += abs(s.mean() - mu) <= 1.96 * se
    return 100 * hits / trials

cases = [
    ("exponential", lambda k: rng_c2.exponential(10, k), 10.0),
    ("lognormal",   lambda k: rng_c2.lognormal(0, 1, k), np.exp(0.5)),
]

for name, f, mu in cases:
    big = f(500_000)
    n_rec = needed_n(big)
    print(f"\n{name}: skew {stats.skew(big):.2f}  ->  recommends n = {n_rec}")
    for n in [30, n_rec, 4*n_rec]:
        print(f"   n = {n:>5}:  coverage {coverage(f, n, mu):5.1f}%")

The rule is a big improvement on "n = 30" and it still **undershoots**.

- **Exponential** (skew 2): recommends n ≈ 45. Coverage at 30 is about 91%, at 45 about 93%,
  and at 180 about 94%.
- **Lognormal** (true skew 6.19, measured here as 5.7): recommends n in the **300s or 400s**,
  depending on which sample you happen to measure the skew from. Coverage at 30 is about 87%,
  at the recommended n about 94%, and 4× more buys nothing further — it is still 94%.

**Does it deliver what it promises? Not quite**, and in two separate ways.

1. **The coverage falls short.** Aiming for a mean-skew of 0.3 gets you to roughly 93–94%, not
   95% — and the lognormal row then *flattens*: quadrupling the recommended n does not close
   the remaining gap. Something other than skew is holding it back, namely the heavy tail
   making the sample's own σ unreliable.
2. **The input is itself an estimate.** Measure the skew of this same lognormal from six
   different samples and you get values from about 5.6 to 7.0 — so the rule recommends anything
   from 350 to 540. It is a guide, not a calculation.

Both caveats point the same way: the rule is a large improvement on "n = 30" because it at
least *scales with the problem*, and it is not a guarantee.

Two honest conclusions:

1. The rule is the right *shape* of answer — it scales with skew², and it tells you that a
   lognormal needs ten times what an exponential needs. That is far more useful than a fixed 30.
2. If you need true 95% coverage on skewed data, no sample size rule will get you there
   cheaply. Use a bootstrap instead (Part D).

### C3 — size an experiment

You are planning an A/B test. Historical conversion is 5%, and you want to detect a lift to 6%.

1. Under the null (both at 5%), what is the standard error of the difference between two
   proportions at n per arm?  *Hint: variances add.*
2. Find the n per arm at which a 1-percentage-point lift sits 2.8 standard errors away.
   (2.8 is roughly what you need for the usual 80% power at 5% significance.)
3. Check your answer by simulating the test 5,000 times and measuring how often it fires.
4. Say what that means in plain English for the team running it.

In [ ]:
rng_c3 = np.random.default_rng(273)
p0, p1 = 0.05, 0.06

#  1. SE of a difference: variances add
def se_diff(n, p=p0):
    return np.sqrt(2 * p * (1 - p) / n)

#  2. solve  (p1 - p0) / se_diff(n) = 2.8
n_needed = 2 * p0 * (1 - p0) * (2.8 / (p1 - p0)) ** 2
print(f"1.  SE of the difference at n=1000: {se_diff(1000):.5f}")
print(f"2.  n needed per arm: {n_needed:,.0f}")

#  3. simulate the whole test
def power(n, trials=5000):
    fires = 0
    for _ in range(trials):
        a = rng_c3.binomial(n, p0) / n
        b = rng_c3.binomial(n, p1) / n
        pooled = (a + b) / 2
        se = np.sqrt(2 * pooled * (1 - pooled) / n)
        fires += abs(b - a) / se > 1.96
    return 100 * fires / trials

print()
for n in [2_000, 5_000, int(n_needed), 20_000]:
    print(f"3.  n = {n:>7,} per arm:  detects the lift {power(n):5.1f}% of the time")

1. SE of the difference is `sqrt(2p(1−p)/n)` — the two arms' variances **add**, which is where
   the 2 comes from.
2. The formula gives about **7,450 per arm**, so roughly **15,000 users** in total.
3. Simulation confirms it: about 80% power at that n, much lower at 2,000, and near-certain at
   20,000.

**In plain English for the team:**

> *"To reliably spot a 5% → 6% improvement we need about 15,000 users, split evenly. At 2,000
> per arm we would miss a real improvement more than half the time — so a negative result from
> a test that size would tell us almost nothing."*

That last sentence is the valuable one, and it is the same point as Day 25's A/B question and
Day 26's loaded die: **a null result is meaningless without the sample size that produced it.**

### C4 — the running-mean diagnostic

Write a function `settles(data)` that returns `True` if the running mean of `data` has settled
down, and `False` if it has not.

1. Scale the wobble by something robust — the IQR works well. (Why not the mean?)
2. Test it on a normal, an exponential, a lognormal and a Cauchy.
3. Plot all four running means on one figure.
4. Which one fails, and what would you report for it instead?

In [ ]:
rng_c4 = np.random.default_rng(274)

def settles(data, threshold=0.10):
    d = np.asarray(data, dtype=float)
    run = np.cumsum(d) / np.arange(1, len(d) + 1)
    tail = run[len(d)//2:]                       # the second half only
    scale = np.percentile(d, 75) - np.percentile(d, 25)
    drift = (tail.max() - tail.min()) / scale
    return drift < threshold, drift

sets = [
    ("normal",      rng_c4.normal(10, 3, 20_000)),
    ("exponential", rng_c4.exponential(10, 20_000)),
    ("lognormal",   rng_c4.lognormal(0, 1, 20_000)),
    ("Cauchy",      stats.cauchy.rvs(size=20_000, random_state=rng_c4) + 50),
]

fig, axes = plt.subplots(1, 4, figsize=(13, 2.6))
for ax, (name, d) in zip(axes, sets):
    ok, drift = settles(d)
    print(f"{name:<13} drift {drift:7.3f} IQRs   {'settles' if ok else 'DOES NOT SETTLE'}")
    run = np.cumsum(d)/np.arange(1, len(d)+1)
    ax.plot(run, color="#2FA98C" if ok else "#E8674C", lw=1.2)
    ax.set_title(name, fontsize=10.5); ax.set_xlabel("n")
plt.tight_layout(); plt.show()

**Why the IQR and not the mean?** Because the mean is exactly the thing that might be
misbehaving. A Cauchy shifted to sit around 50 has a large mean, so dividing by it would make
even wild swings look like a small percentage. The IQR is a robust measure of the data's own
scale and does not care about the tails.

**Results:** the normal, exponential and lognormal all drift by about **0.01 IQRs**. The Cauchy
drifts by about **1.9** — more than a hundred times further. The separation is not subtle, which
is what makes this a usable diagnostic rather than a judgement call.

**What to report for the Cauchy instead:** the **median**, which is perfectly well-behaved for a
Cauchy, with its uncertainty from a bootstrap. And say explicitly that the mean is not a
meaningful summary of this data, rather than quietly reporting one.

### C5 — a real decision

A delivery company's times are lognormal-ish: median 25 minutes, with a long tail. They want to
advertise an average delivery time with a ±1 minute margin.

1. Simulate a realistic population and report its mean, median and skew.
2. How large a sample do they need for a standard error of 0.5 minutes (so ±1 minute)?
3. Check the real coverage at that n. Do they get the ±1 they advertised?
4. What would you actually advise them to advertise, and why?

In [ ]:
rng_c5 = np.random.default_rng(275)
pop = rng_c5.lognormal(np.log(25), 0.55, 500_000)
true_mean = pop.mean()

print(f"1.  mean {true_mean:.2f}  median {np.median(pop):.2f}  "
      f"sd {pop.std():.2f}  skew {stats.skew(pop):.2f}")
print(f"    90th percentile {np.percentile(pop, 90):.1f} min")

#  2. n = (sigma / SE)^2
n_needed = (pop.std() / 0.5) ** 2
print(f"\n2.  n for SE = 0.5:  {n_needed:,.0f}")

#  3. does it deliver?
print()
for n in [int(n_needed//4), int(n_needed), int(n_needed*4)]:
    hits = width = 0
    for _ in range(3000):
        s = rng_c5.choice(pop, n)
        se = s.std(ddof=1)/np.sqrt(n)
        hits  += abs(s.mean() - true_mean) <= 1.96*se
        width += 1.96*se
    print(f"3.  n = {n:>6,}: coverage {100*hits/3000:5.1f}%   "
          f"average half-width +/-{width/3000:.2f} min")

1. Mean about **29 minutes**, median **25**, skew about **2.1**. The mean sits well above the
   median — the long tail of slow deliveries drags it up.
2. With σ ≈ 17, a standard error of 0.5 needs n = (17/0.5)² ≈ **1,200 deliveries**.
3. At that n the coverage is close to 95% and the half-width is about ±1 minute, so the
   arithmetic works. (The skew rule agrees: (2.1/0.3)² ≈ 49, comfortably below 1,200.)
4. **But advertising the mean is the wrong decision anyway.**

The mean of 29 minutes describes no one's experience. The typical delivery takes 25 minutes,
and 10% take over 50. A customer who waits 50 minutes will not be consoled by a correctly
computed average.

**What to advise:** advertise a **percentile**, not a mean — *"90% of deliveries arrive within
51 minutes"* is a promise customers can check and the company can be held to. Then use the
confidence interval internally, to track whether that percentile is drifting.

This is the Day 25 call-centre lesson in a new costume: the CLT tells you how precisely you
know the average, and knowing the average precisely is not the same as it being the right
number to quote.

---
# Part D — Challenge

### D1 — break the theorem, then rescue it

Build a dataset where a textbook confidence interval is badly wrong, demonstrate that it is
wrong, and then produce a correct interval anyway.

1. Construct a population where a standard 95% interval at n = 50 covers well under 95%.
   (A heavy right tail is the easiest route.)
2. Measure the real coverage by simulation, and say how bad it is.
3. Build a **bootstrap percentile interval** instead: resample the sample *with replacement*
   2,000 times, take the mean of each resample, and use the 2.5th and 97.5th percentiles.
4. Measure the bootstrap's coverage at the same n. Is it better?
5. State honestly where the bootstrap still falls short.

In [ ]:
rng_d1 = np.random.default_rng(2701)

#  1. a nasty but entirely realistic population: lognormal, skew ~11
pop = rng_d1.lognormal(0, 1.2, 400_000)
mu  = pop.mean()
print(f"population: mean {mu:.3f}  median {np.median(pop):.3f}  skew {stats.skew(pop):.2f}")
print(f"  n the skew rule asks for: {(stats.skew(pop)/0.3)**2:,.0f}")

In [ ]:
#  2 & 3. the two methods, side by side
def classic_ci(s):
    se = s.std(ddof=1)/np.sqrt(len(s))
    return s.mean() - 1.96*se, s.mean() + 1.96*se

def bootstrap_ci(s, B=2000, rng=None):
    rng = rng or np.random.default_rng()
    idx  = rng.integers(0, len(s), (B, len(s)))
    boot = s[idx].mean(axis=1)
    return np.percentile(boot, 2.5), np.percentile(boot, 97.5)

def compare(n, trials=1500):
    c_hits = b_hits = 0
    c_w = b_w = 0.0
    for _ in range(trials):
        s = rng_d1.choice(pop, n)
        lo, hi = classic_ci(s)
        c_hits += lo <= mu <= hi; c_w += hi - lo
        lo, hi = bootstrap_ci(s, 1000, rng_d1)
        b_hits += lo <= mu <= hi; b_w += hi - lo
    return (100*c_hits/trials, c_w/trials, 100*b_hits/trials, b_w/trials)

print(f"{'n':>6}  {'classic':>18}  {'bootstrap':>18}")
print(f"{'':>6}  {'cover':>8}{'width':>10}  {'cover':>8}{'width':>10}")
for n in [20, 50, 200, 1000]:
    c, cw, b, bw = compare(n)
    print(f"{n:>6}  {c:>7.1f}%{cw:>10.2f}  {b:>7.1f}%{bw:>10.2f}")

In [ ]:
#  5. where does the bootstrap still fall short? Try it on infinite variance.
heavy = (rng_d1.pareto(1.3, 300_000) + 1)      # mean finite, variance infinite
mu_h  = heavy.mean()
print(f"Pareto(1.3): sample mean {mu_h:.2f}, variance is INFINITE in theory\n")

for n in [50, 500]:
    c_hits = b_hits = 0
    for _ in range(1200):
        s = rng_d1.choice(heavy, n)
        lo, hi = classic_ci(s);              c_hits += lo <= mu_h <= hi
        lo, hi = bootstrap_ci(s, 800, rng_d1); b_hits += lo <= mu_h <= hi
    print(f"n = {n:>4}:  classic {100*c_hits/1200:5.1f}%   bootstrap {100*b_hits/1200:5.1f}%")

**2. How bad is the classic interval?** On a lognormal with σ_log = 1.2 (measured skew around
12), a "95%" interval covers about **81%** at n = 20 and **87%** at n = 50. Roughly one survey
in seven is wrong at n = 50, with no warning. It only reaches 94% at n = 1000.

**4. Is the bootstrap better?** **Barely.** And this is the honest answer the exercise was
built to produce:

| n | classic | bootstrap |
|---|---|---|
| 20 | 80.6% | 81.5% |
| 50 | 86.7% | 87.3% |
| 200 | 92.7% | 92.9% |
| 1000 | 94.3% | **93.8%** |

The bootstrap wins by well under a percentage point where it wins at all — and at n = 1000 it
is slightly *worse*. Its intervals are marginally narrower, so it is not buying its small
advantage by being vague, but the advantage is close to nothing.

If you expected the bootstrap to rescue this, that expectation is the thing to correct. It is
widely described as the answer to non-normality, and on a badly skewed mean at small n it is
**not** a rescue.

**5. Where the bootstrap falls short:**

- **It does not restore the missing coverage.** At n = 50 both methods sit in the mid-80s. The
  bootstrap is a different approximation, not a better guarantee.
- **It inherits your sample's blind spots.** A bootstrap resamples the data you have. With a
  heavy right tail, a sample of 50 will usually miss the extreme values entirely — and then
  every one of the 2,000 resamples misses them too. The bootstrap cannot invent data you did
  not collect, and for exactly this reason it tends to be over-confident in precisely the
  situation you wanted help with.
- **It fails completely on infinite variance.** On a Pareto(1.3) both methods collapse to
  around **34%** coverage. A "95%" interval that is right one time in three is worse than
  useless, and the bootstrap does not notice.

**The useful conclusion:** when you need the mean of a heavy-tailed quantity, the bootstrap is
not the escape hatch. Your real options are a much larger sample, or reporting a different
statistic — the **median**, which was well behaved in every case above. Choosing a better
statistic beats computing a cleverer interval for a bad one.

### D2 — audit a published claim

A press release reports:

> *"In a study of 40 participants, our supplement reduced average recovery time from 48 hours
> to 41 hours — a 15% improvement (p < 0.05)."*

Recovery times are strongly right-skewed: most people recover quickly, a few take much longer.

1. Simulate a plausible population of recovery times with a mean of 48 hours and a realistic
   skew.
2. With n = 40 per group and **no real effect at all**, how often would a test like this report
   a significant difference? And when it does, how large does the "improvement" look?
3. Work out what n the study would have needed, using both the skew rule and a direct
   simulation.
4. Write two or three sentences you would send back to the authors.

Be careful to distinguish two separate problems: the test may be firing too often, *and* the
reported effect size may be inflated even when it is right.

In [ ]:
rng_d2 = np.random.default_rng(2702)

#  1. recovery times: lognormal, mean 48 hours, strongly right-skewed
sig = 0.8
mu_log = np.log(48) - sig**2/2
pop = rng_d2.lognormal(mu_log, sig, 400_000)
print(f"mean {pop.mean():.1f}h   median {np.median(pop):.1f}h   "
      f"skew {stats.skew(pop):.2f}   90th pct {np.percentile(pop,90):.0f}h")
print(f"n the skew rule asks for: {(stats.skew(pop)/0.3)**2:,.0f} per group")

In [ ]:
#  2. NO real effect. Both groups from the same population. How often does it fire?
n, trials = 40, 20_000
fires, lifts = 0, []
for _ in range(trials):
    a = rng_d2.choice(pop, n)
    b = rng_d2.choice(pop, n)
    if stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05:
        fires += 1
        lifts.append((a.mean() - b.mean()) / a.mean())

lifts = np.abs(np.array(lifts)) * 100
print(f"false positive rate at n={n}: {100*fires/trials:.1f}%   (should be 5.0%)")
print()
print("WHEN it fires by pure chance, the 'improvement' it reports:")
print(f"  median {np.median(lifts):.1f}%    10th-90th pct "
      f"{np.percentile(lifts,10):.0f}% to {np.percentile(lifts,90):.0f}%")
print(f"  share of false positives showing >= 15%: {(lifts >= 15).mean():.1%}")

In [ ]:
#  3. what n would be needed - two routes
print(f"skew rule:      {(stats.skew(pop)/0.3)**2:,.0f} per group")
print()
print("direct simulation of the false-positive rate:")
for n in [40, 100, 300, 1000]:
    fires = sum(stats.ttest_ind(rng_d2.choice(pop, n), rng_d2.choice(pop, n),
                                equal_var=False).pvalue < 0.05
                for _ in range(6000))
    print(f"  n = {n:>5}:  fires {100*fires/6000:4.1f}% of the time with NO effect")

In [ ]:
#  and if the effect were real - could n=40 even detect it reliably?
treated = pop * (41/48)          # a genuine 15% reduction
power = sum(stats.ttest_ind(rng_d2.choice(pop, 40), rng_d2.choice(treated, 40),
                            equal_var=False).pvalue < 0.05
            for _ in range(6000))
print(f"with a REAL 15% effect, n=40 detects it {100*power/6000:.0f}% of the time")
for n in [100, 300, 600]:
    p = sum(stats.ttest_ind(rng_d2.choice(pop, n), rng_d2.choice(treated, n),
                            equal_var=False).pvalue < 0.05
            for _ in range(4000))
    print(f"  n = {n:>4}: {100*p/4000:.0f}%")

**1.** A lognormal with σ_log = 0.8 gives a mean of 48 hours, a median around **35**, a skew
near **3.6**, and a 90th percentile around **97 hours**. That is a realistic shape for recovery
data, and note immediately that the mean of 48 is nowhere near the typical experience of 35.

**2.** This is where the two problems separate, and they point in opposite directions.

- **The test itself is honest.** With n = 40 per group and no effect at all, Welch's t-test
  fires about **4.8%** of the time — essentially the 5% it promises. Welch's test is far more
  robust to skew than the raw CLT is, and that is worth saying out loud rather than assuming
  everything is broken.
- **But the effect size it reports is wildly inflated.** When it fires by pure chance, the
  median apparent "improvement" is **43%**, and **every single false positive** shows at least
  15%. That is the finding. A 15% improvement is not merely *consistent* with a false positive
  at this sample size — it is at the small end of what a false positive looks like.

This is the "winner's curse" of underpowered studies: the only way a small study reaches
significance is by landing on an unusually large apparent effect, so the effects that get
published are systematically overstated.

**3.** Two routes, and they disagree in an informative way.

- The **skew rule** asks for about **148 per group** for the mean itself to be well behaved.
- **Simulation of the false-positive rate** says 40 is already fine — it fires at 4.3%.
- **Simulation of the power** says 40 is nowhere near enough: with a genuinely real 15% effect,
  n = 40 detects it only **11%** of the time. n = 300 gets you to 54%, and you need around
  **600 per group** for the usual 80%.

So the binding constraint is not the CLT at all — it is **power**. That is worth noticing: the
tool you reach for first is not always the one that answers the question.

**4. What I would send back:**

> *"Two concerns. First, the study is severely underpowered: if your 15% effect is real, a
> trial with 40 per group would detect it only about one time in nine, so a significant result
> here is more likely to be noise than signal. Second — and this follows from the first — when
> we simulate your exact design with no effect whatsoever, every false positive it produces
> shows an apparent improvement of at least 15%, with a median of 43%. A 15% figure is
> therefore not evidence of a small real effect; it is what chance looks like at this sample
> size. We would want around 600 per group, and we would suggest reporting median recovery time
> alongside the mean, since the mean of 48 hours is pulled up by a few very slow recoveries and
> describes no typical participant."*

**The thing to notice:** p < 0.05 was satisfied, the test was correctly calibrated, and the
conclusion is still unreliable. "Statistically significant" and "trustworthy" are different
claims, and the gap between them is mostly about sample size — which is what Day 28 picks up.

---
# Where the marks were

| Question | The thing being tested |
|---|---|
| A1 | skew of the mean = population skew / √n, exactly |
| A2 | ⚠️ σ describes people; SE describes the average |
| A3 | n = (σ/SE)² — the √n law as a budget |
| A4 | infinite variance: averaging buys nothing |
| A5 | testing the data for normality tests the wrong thing |
| A6 | variances add; that is where √n comes from |
| A7 | ⚠️ ÷√n, not ÷n — and the error grows with n |
| A8 | every margin of error is σ/√n in disguise |
| B1 | the ÷n slip, caught by a plausibility check |
| B2 | a CI for the mean is not a range for individuals |
| B3 | dependence, and effective sample size |
| B4 | "n > 30" does not rescue infinite variance |
| C1 | non-normal is not only about skew — watch kurtosis too |
| C2 | the skew rule helps, and still undershoots |
| C3 | sizing an experiment, and what a null result is worth |
| C4 | a robust diagnostic, and why the IQR not the mean |
| C5 | knowing the mean precisely ≠ the mean being the right number |
| D1 | the bootstrap: better, and not a free lunch |
| D2 | significance and trustworthiness are different claims |

If you only remember three things from today:

1. **The theorem is about the MEAN, never about your data.** Your data stays exactly as skewed
   as it always was.
2. **σ/√n.** Divide by the square root. Four times the data for twice the precision.
3. **Check before you lean on it:** finite σ, independent observations, and n ≈ (skew/0.3)² —
   not 30.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*